In [13]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
import os
import torch
from torch.utils.data import Dataset
import torchvision
import torchvision.transforms as transforms
import pandas as pd
from vit import ViTDecoder_v3 as vit3, ViTDecoder_v2 as vit
from PIL import Image
import math as math
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models
import numpy as np
import matplotlib.pyplot as plt
import time

In [2]:
# Custom Padding Transformer (FIX THIS)
class load_csv_as_image:
    def __call__(self, path):
        df = pd.read_csv(path, header=None)
        data = df.values  # or use np.loadtxt for speed
        rows, cols = df.shape
        return torch.tensor(data).reshape((rows,cols)).float()

class PadToSize:
    def __init__(self, target):
        self.target = target

    def __call__(self, img):
        # Calculate padding sizes

        h, w = img.shape
        target_w, target_h = self.target

        pad_w = max(0, target_w - w)
        pad_h = max(0, target_h - h)

        pad_left = pad_w // 2
        pad_right = pad_w - pad_left
        pad_top = pad_h // 2
        pad_bot = pad_h - pad_top

        padding = (pad_left, pad_top, pad_right, pad_bot)
        img1 = transforms.functional.pad(img, padding, fill=0, padding_mode='reflect')
        return img1
        
class ConvertToFloat32(object):
    def __call__(self, tensor):
        return tensor.to(torch.float32)

class AddChannelDim:
    def __call__(self, x):
        return x.unsqueeze(0)  # Adds channel at dim 0
    
class normalize_min_max():
    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val

    def __call__(self, x):
        return (x - self.min_val) / (self.max_val - self.min_val + 1e-8)

In [3]:
target_size = (930, 930)
pipeline= transforms.Compose([
    #transforms.ToTensor(),  # Convert image to tensor
    load_csv_as_image(),
    # transforms.Lambda(lambda x: x[:3]),
    PadToSize(target_size),  # Pad the image to the target size
    ConvertToFloat32(),
    AddChannelDim()
    #transforms.ToPILImage(), 
])

In [4]:
class NetlistDataset(Dataset):
    def __init__(self, image_dir, netlist_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.netlist_dir = netlist_dir
        self.image_files = sorted(os.listdir(image_dir))  # Sorted to align image order
        self.label_files = sorted(os.listdir(label_dir))  # Assuming labels have the same order
        self.netlist_files = sorted(os.listdir(netlist_dir))  # Assuming labels have the same order
        self.transform = transform
        self.current_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=9.010990000000002e-06)
            ])
        self.eff_dist_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=61.3528)  # Normalize
            ])
        self.pdn_density_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=3)   # Normalize
            ])
        self.ir_drop_transform = transforms.Compose([
            normalize_min_max(min_val=6.58885e-05, max_val=0.00609974)   # Normalize
            ])

    
    def __len__(self):
        return len(self.label_files) # based off of the label amount
    
    def __getitem__(self, idx):
        # Get 3 consecutive images for stacking (you could choose any other strategy here)
        names = []
        stacked_images = []

        img1_path = os.path.join(self.image_dir, self.image_files[3*idx])
        img2_path = os.path.join(self.image_dir, self.image_files[3*idx+1])
        img3_path = os.path.join(self.image_dir, self.image_files[3*idx+2])
        
        netlist1_path = os.path.join(self.netlist_dir, self.netlist_files[4*idx])
        netlist2_path = os.path.join(self.netlist_dir, self.netlist_files[4*idx+1])
        netlist3_path = os.path.join(self.netlist_dir, self.netlist_files[4*idx+2])
        netlist4_path = os.path.join(self.netlist_dir, self.netlist_files[4*idx+3])

        # Load images
        img1 = self.transform(img1_path)
        img2 = self.transform(img2_path)
        img3 = self.transform(img3_path)
        # if self.transform:
        img1 = self.current_transform(img1)
        img2 = self.eff_dist_transform(img2)
        img3 = self.pdn_density_transform(img3)

        # load netlists
        netlist1 = self.transform(netlist1_path)
        netlist2 = self.transform(netlist2_path)
        netlist3 = self.transform(netlist3_path)
        netlist4 = self.transform(netlist4_path)

        stacked_net = torch.concat([netlist1, netlist2, netlist3, netlist4])

        # Stack the 3 images into a list for separate encoding
        stacked_images.append(img1)
        stacked_images.append(img2)
        stacked_images.append(img3)
        stacked_images.append(stacked_net)

        # Load corresponding label image
        label_path = os.path.join(self.label_dir, self.label_files[idx])  # Label corresponding to last image in stack
        label = self.transform(label_path)
        # if self.transform:
        label = self.ir_drop_transform(label)
        
        names.append(self.image_files[3*idx])
        names.append(self.image_files[3*idx+1])
        names.append(self.image_files[3*idx+2])
        names.append(self.label_files[idx])

        return stacked_images, label, names #, idx, names

In [5]:
class StackedImagesDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = sorted(os.listdir(image_dir))  # Sorted to align image order
        self.label_files = sorted(os.listdir(label_dir))  # Assuming labels have the same order
        self.transform = transform
        self.current_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=9.010990000000002e-06)
            ])
        self.eff_dist_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=61.3528)  # Normalize
            ])
        self.pdn_density_transform = transforms.Compose([
            normalize_min_max(min_val=0, max_val=3)   # Normalize
            ])
        self.ir_drop_transform = transforms.Compose([
            normalize_min_max(min_val=6.58885e-05, max_val=0.00609974)   # Normalize
            ])

    
    def __len__(self):
        return len(self.label_files) # based off of the label amount
    
    def __getitem__(self, idx):
        # Get 3 consecutive images for stacking (you could choose any other strategy here)
        names = []
        stacked_images = []

        img1_path = os.path.join(self.image_dir, self.image_files[3*idx])
        img2_path = os.path.join(self.image_dir, self.image_files[3*idx+1])
        img3_path = os.path.join(self.image_dir, self.image_files[3*idx+2])
        
        # Load images
        img1 = self.transform(img1_path)
        img2 = self.transform(img2_path)
        img3 = self.transform(img3_path)
        # if self.transform:
        img1 = self.current_transform(img1)
        img2 = self.eff_dist_transform(img2)
        img3 = self.pdn_density_transform(img3)

        # Stack the 3 images into a list for separate encoding
        stacked_images.append(img1)
        stacked_images.append(img2)
        stacked_images.append(img3)

        # Load corresponding label image
        label_path = os.path.join(self.label_dir, self.label_files[idx])  # Label corresponding to last image in stack
        label = self.transform(label_path)
        # if self.transform:
        label = self.ir_drop_transform(label)
        
        names.append(self.image_files[3*idx])
        names.append(self.image_files[3*idx+1])
        names.append(self.image_files[3*idx+2])
        names.append(self.label_files[idx])

        return stacked_images, label, names #, idx, names

In [8]:
netlist_dataset = NetlistDataset(image_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/EvaluationData/evaluation_inputs", netlist_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/EvaluationData/evaluation_netlists", label_dir= "/home/bqtx/Documents/VLSI/ir_drop_ml/EvaluationData/evaluation_ir_drop", transform=pipeline)
eval_dataset = StackedImagesDataset(image_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/EvaluationData/evaluation_inputs", label_dir= "/home/bqtx/Documents/VLSI/ir_drop_ml/EvaluationData/evaluation_ir_drop", transform=pipeline)

In [9]:
eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=False)
netlist_dataloader = DataLoader(netlist_dataset, batch_size=1, shuffle=False)

Original Model Metrics Calc:

In [7]:
original_model = vit(image_size = (930,930), patch_size = (15,15), dim = 768, depth = 1, heads = 4, mlp_dim = 10, channels=1, out_channels=1)

original_model.load_state_dict(torch.load("/home/bqtx/Documents/VLSI/ir_drop_ml/model/0409_weights.pth", weights_only=True))
original_model.eval()

ViTDecoder_v2(
  (to_patch_embedding_X): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=15, p2=15)
    (1): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=225, out_features=768, bias=True)
    (3): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (dropout_X): Dropout(p=0.0, inplace=False)
  (transformer_X): Transformer(
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0): ModuleList(
        (0): Attention(
          (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (dropout): Dropout(p=0.0, inplace=False)
          (to_qkv): Linear(in_features=768, out_features=768, bias=False)
          (to_out): Sequential(
            (0): Linear(in_features=256, out_features=768, bias=True)
            (1): Dropout(p=0.0, inplace=False)
          )
        )
        (1): FeedForward(
          (net): Sequential(
     

In [17]:
original_model.eval()  # Set the model to evaluation mode
criterion = nn.L1Loss()  # For multi-class classification
optimizer = optim.Adam(original_model.parameters(), lr=0.001, weight_decay=1e-5)
running_loss = 0.0
first = 0
current_time = time.time()
# print(current_time)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():  # No need to track gradients during evaluation
    for batch_idx, (inputs, labels, names) in enumerate(tqdm(eval_dataloader, desc=f"Testing Epoch", ncols=100)):
        labels = labels.to(device)
        
        new_inp = []
        for inp in inputs:
            new_inp.append(inp.to(device))
        outputs = original_model(inputs, names)
        loss = criterion(outputs, labels)  # Calculate the loss
        running_loss += loss.item()

end_time = time.time()

print(f"Test Loss: {running_loss/len(eval_dataloader):.6f}")
print(f"Avg. Inference Time: {(end_time-current_time)/len(eval_dataloader):.6f}")
print("-" * 50)

Testing Epoch: 100%|████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.15it/s]

Test Loss: 0.157972
Avg. Inference Time: 0.872958
--------------------------------------------------


With Vias

In [18]:
via_model = vit3(image_size = (930,930), patch_size = (15,15), dim = 768, depth = 1, heads = 4, mlp_dim = 10, channels=1, out_channels=1)

via_model.load_state_dict(torch.load("/home/bqtx/Documents/VLSI/ir_drop_ml/model/0419_weights.pth", weights_only=True))
via_model.eval()

ViTDecoder_v3(
  (to_patch_embedding_X): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=15, p2=15)
    (1): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=225, out_features=768, bias=True)
    (3): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (dropout_X): Dropout(p=0.0, inplace=False)
  (transformer_X): Transformer(
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0): ModuleList(
        (0): Attention(
          (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (dropout): Dropout(p=0.0, inplace=False)
          (to_qkv): Linear(in_features=768, out_features=768, bias=False)
          (to_out): Sequential(
            (0): Linear(in_features=256, out_features=768, bias=True)
            (1): Dropout(p=0.0, inplace=False)
          )
        )
        (1): FeedForward(
          (net): Sequential(
     

In [19]:
via_model.eval()  # Set the model to evaluation mode
criterion = nn.L1Loss()  # For multi-class classification
optimizer = optim.Adam(via_model.parameters(), lr=0.001, weight_decay=1e-5)
running_loss = 0.0
first = 0
current_time = time.time()
# print(current_time)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():  # No need to track gradients during evaluation
    for batch_idx, (inputs, labels, names) in enumerate(tqdm(netlist_dataloader, desc=f"Testing Epoch", ncols=100)):
        labels = labels.to(device)
        
        new_inp = []
        for inp in inputs:
            new_inp.append(inp.to(device))
        outputs = via_model(inputs, names)
        loss = criterion(outputs, labels)  # Calculate the loss
        running_loss += loss.item()

end_time = time.time()

print(f"Test Loss: {running_loss/len(eval_dataloader):.6f}")
print(f"Avg. Inference Time: {(end_time-current_time)/len(netlist_dataloader):.6f}")
print("-" * 50)

Testing Epoch: 100%|████████████████████████████████████████████████| 10/10 [00:11<00:00,  1.14s/it]

Test Loss: 0.160057
Avg. Inference Time: 1.141638
--------------------------------------------------


Trained on Augmented Data

In [23]:
aug_model = vit(image_size = (930,930), patch_size = (15,15), dim = 768, depth = 1, heads = 4, mlp_dim = 10, channels=1, out_channels=1)

aug_model.load_state_dict(torch.load("/home/bqtx/Documents/VLSI/ir_drop_ml/model/0421_weights.pth", weights_only=True))
aug_model.eval()

ViTDecoder_v2(
  (to_patch_embedding_X): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=15, p2=15)
    (1): LayerNorm((225,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=225, out_features=768, bias=True)
    (3): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (dropout_X): Dropout(p=0.0, inplace=False)
  (transformer_X): Transformer(
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0): ModuleList(
        (0): Attention(
          (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (dropout): Dropout(p=0.0, inplace=False)
          (to_qkv): Linear(in_features=768, out_features=768, bias=False)
          (to_out): Sequential(
            (0): Linear(in_features=256, out_features=768, bias=True)
            (1): Dropout(p=0.0, inplace=False)
          )
        )
        (1): FeedForward(
          (net): Sequential(
     

In [24]:
aug_model.eval()  # Set the model to evaluation mode
criterion = nn.L1Loss()  # For multi-class classification
optimizer = optim.Adam(aug_model.parameters(), lr=0.001, weight_decay=1e-5)
running_loss = 0.0
first = 0
current_time = time.time()
# print(current_time)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():  # No need to track gradients during evaluation
    for batch_idx, (inputs, labels, names) in enumerate(tqdm(eval_dataloader, desc=f"Testing Epoch", ncols=100)):
        labels = labels.to(device)
        
        new_inp = []
        for inp in inputs:
            new_inp.append(inp.to(device))
        outputs = aug_model(inputs, names)
        loss = criterion(outputs, labels)  # Calculate the loss
        running_loss += loss.item()

end_time = time.time()

print(f"Test Loss: {running_loss/len(eval_dataloader):.6f}")
print(f"Avg. Inference Time: {(end_time-current_time)/len(eval_dataloader):.6f}")
print("-" * 50)

Testing Epoch: 100%|████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.14it/s]

Test Loss: 0.161546
Avg. Inference Time: 0.876084
--------------------------------------------------
